In [ ]:
from openpyxl import load_workbook
from pandas import ExcelWriter
import numpy as np
import pandas as pd
import requests

In [ ]:
p = 'https://www.ons.gov.uk/file?uri=%2fbusinessindustryandtrade%2fretailindustry%2fdatasets%2fretailsalesindexinternetsales%2fcurrent/internetsalesreferencetables.xlsx'
g = 'https://www.ons.gov.uk/file?uri=%2fbusinessindustryandtrade%2fretailindustry%2fdatasets%2fpoundsdatatotalretailsales%2fcurrent/poundsdataaccessible.xlsx'

with open(f"ons_xlsx_total.xlsx", "wb") as f:
    f.write(requests.get(g).content)

with open(f"ons_xlsx_online.xlsx", "wb") as f:
    f.write(requests.get(p).content)

In [ ]:
def format_ons_xlsx_online():
    wb = load_workbook('ons_xlsx_online.xlsx')
    ws = wb['IntValSA'].values
    x = []
    df = pd.DataFrame(ws).iloc[3:].reset_index(drop=True)
    header = df.iloc[0]
    df = df[1:]
    df.columns = header
    df = df.dropna(axis='columns').iloc[2:].reset_index(drop=True)
    df['Time Period'] = pd.to_datetime(df['Time Period'], format='%d%b%Y:%H:%M:%S.%f')
    df = df.set_index('Time Period')
    df = df * (52/12)
    df.columns = (df.columns.str.split('[')).str[0]
    df.columns = map(str.lower, df.columns)
    df.columns = df.columns.str.strip()
    # df.to_excel('ons_xlsx_online-edited.xlsx')
    return pd.DataFrame(df)

format_ons_xlsx_online()

In [ ]:

def format_ons_xlsx_total():
    wb = load_workbook('ons_xlsx_total.xlsx')
    ws = wb['ValSAT'].values
    x = []
    df = pd.DataFrame(ws).iloc[4:].reset_index(drop=True)
    header = df.iloc[0]
    df = df[171:]
    df.columns = header
    df = df.drop(['Automotive fuel','All retailing including automotive fuel','Month as a % of Total','Month as a % of Total excluding automotive fuel', 'Total Annual Sales for All Retailing including automotive fuel','Total Annual Sales for All Retailing excluding automotive fuel'], axis=1)
    df = df.rename(columns={df.columns[0]: 'Time Period'}).reset_index(drop=True)
    df = df.rename(columns={df.columns[3]: 'Total of predominantly non-food stores'})
    df['Time Period'] = pd.to_datetime(df['Time Period'])
    df = df.set_index('Time Period')
    df = df.dropna()
    df = df / 1000
    df.columns = map(str.lower, df.columns)
    df.columns = df.columns.str.strip()
    # df.to_excel('ons_xlsx_total-edited.xlsx')
    return pd.DataFrame(df)

format_ons_xlsx_total()

In [ ]:
def format_ons_xlsx_offline():
    return format_ons_xlsx_total() - format_ons_xlsx_online()

def oao_process(format1):
    format1['yoy % all retailing excluding automotive fuel'] = format1['all retailing excluding automotive fuel'].pct_change(periods=12) * 100
    return format1[['yoy % all retailing excluding automotive fuel','predominantly food stores','total of predominantly non-food stores','non-store retailing']].dropna()

def ovo_process(format1):
    format1['yoy % all retailing excluding automotive fuel'] = format1['all retailing excluding automotive fuel'].pct_change(periods=12) * 100
    return format1[['yoy % all retailing excluding automotive fuel','all retailing excluding automotive fuel',]].dropna()

x = format_ons_xlsx_total()
x = x[:].pct_change(periods=12) * 100
x = x[['all retailing excluding automotive fuel','predominantly food stores','total of predominantly non-food stores','non-store retailing']].dropna()

with pd.ExcelWriter('output.xlsx') as writer:  
    format_ons_xlsx_total().to_excel(writer, sheet_name='TotalValSAT')
    format_ons_xlsx_online().to_excel(writer, sheet_name='IntValSAT')
    format_ons_xlsx_offline().to_excel(writer, sheet_name='OfflineValSAT')
    oao_process(format_ons_xlsx_online()).to_excel(writer, sheet_name='online_by_category-flourish')
    oao_process(format_ons_xlsx_offline()).to_excel(writer, sheet_name='offline_by_category-flourish')
    ovo_process(format_ons_xlsx_online()).to_excel(writer, sheet_name='online_yoy_total-flourish')
    ovo_process(format_ons_xlsx_offline()).to_excel(writer, sheet_name='offline_yoy_total-flourish')
    x.to_excel(writer, sheet_name='yoy-by-category_total-flourish')